# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A content item is worth refreshing if it previously performed well but is now stale and declining.
We look for high-traffic items that haven't been updated in months and show slipping position or engagement.

Reason codes:
stale_high_traffic: Old content with historically strong performance
stale_declining : Old content with dropping engagement
position_slip: Previously ranked well, now dropping lower
engagement_drop: Had good engagement, now declining

In [3]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv(r"d:\git\Assignment-1\data\raw\content_refresh_anonymized.csv")

df['signal_stale'] = (df['days_since_last_update'] >= 180).astype(int)
df['signal_previously_strong'] = (df['impressions_90d'] >= 500).astype(int)
df['signal_declining'] = (df['trend_direction'] == 'down').astype(int)
df['signal_position_slip'] = (df['avg_position'] > 0) & (df['avg_position'] >= 20).astype(int)

df['baseline_score'] = (
    df['signal_stale'] * df['signal_previously_strong'] * df['signal_declining'] * df['impressions_90d']
)

def assign_reason_code(row):
    if row['signal_stale'] and row['signal_declining']:
        return 'stale_declining'
    elif row['signal_stale'] and row['signal_previously_strong']:
        return 'stale_high_traffic'
    elif row['signal_declining']:
        return 'engagement_drop'
    elif row['signal_position_slip']:
        return 'position_slip'
    else:
        return 'no_signal'

df['reason_code'] = df.apply(assign_reason_code, axis=1)

# Ranking by score
ranked = df.sort_values('baseline_score', ascending=False).reset_index(drop=True)
ranked['rank'] = range(1, len(ranked) + 1)

# Selecting columns for output
output = ranked[['rank', 'content_id', 'baseline_score', 'reason_code', 
                 'impressions_90d', 'days_since_last_update', 'trend_direction', 'avg_position']]

# Create output directory if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

output.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Baseline written: {len(output)} items")
print("\nTop 10 by score:")
print(output.head(10))

Baseline written: 30000 items

Top 10 by score:
   rank            content_id  baseline_score      reason_code  \
0     1  content_cf56e2e2e282           61678  stale_declining   
1     2  content_7368877ea310           59472  stale_declining   
2     3  content_1bfaa38ff26c           25715  stale_declining   
3     4  content_0a91db491d14           13299  stale_declining   
4     5  content_5feee3994adb            7812  stale_declining   
5     6  content_c2d929d83eaa            7558  stale_declining   
6     7  content_b16bd7307b39            4590  stale_declining   
7     8  content_fe16a55cd13d            4556  stale_declining   
8     9  content_ecb6215e79fd            4429  stale_declining   
9    10  content_928af3e22c80            1697  stale_declining   

   impressions_90d  days_since_last_update trend_direction  avg_position  
0            61678                     194            down          19.7  
1            59472                     194            down          24.8  


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.